In [0]:

from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg, round as spark_round,
    countDistinct, when, lit, coalesce, current_timestamp
)

# STEP 1: CREATE DIMENSION TABLES

print("Creating Dimension Tables\n")

# Dimension: Customers
print(" dim_customers")
df_dim_customers = spark.table("globaldistributor.02_silver.customers") \
    .select(
        "customer_id",
        "customer_name",
        "customer_type",
        "country",
        "signup_date"
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("customer_id")

df_dim_customers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.dim_customers")

print(f"{df_dim_customers.count()} rows")

# Dimension: Products
print("dim_products")
df_dim_products = spark.table("globaldistributor.02_silver.products") \
    .select(
        "product_id",
        "product_name",
        "category",
        "cost_price"
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("product_id")

df_dim_products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.dim_products")

print(f"{df_dim_products.count()} rows")

# Dimension: Regions
print("dim_regions")
df_dim_regions = spark.table("globaldistributor.02_silver.regions") \
    .select(
        "region_code",
        "region_name",
        "country"
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("region_code")

df_dim_regions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.dim_regions")

print(f"{df_dim_regions.count()} rows")

# Dimension: Invoices
print("dim_invoices")
df_dim_invoices = spark.table("globaldistributor.02_silver.invoices") \
    .select(
        "invoice_id",
        "customer_id",
        "invoice_date",
        "invoice_month",
        "currency",
        "region",
        "invoice_status"
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("invoice_id")

df_dim_invoices.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.dim_invoices")

print(f"{df_dim_invoices.count()} rows")

In [0]:

# STEP 2: CREATE FACT TABLE


print("Creating Fact Table\n")

df_lines = spark.table("globaldistributor.02_silver.invoice_line_items")
df_invoices = spark.table("globaldistributor.02_silver.invoices")

print("fact_sales ")
# Join line items with invoices to get complete sales facts
df_fact_sales = df_lines \
    .join(df_invoices, "invoice_id", "inner") \
    .select(
        df_lines["invoice_line_id"].alias("sales_id"),  # Primary key
        df_lines["invoice_id"],
        df_lines["product_id"],                          #  to dim_products
        df_invoices["customer_id"],                      #  to dim_customers
        df_invoices["region"].alias("region_code"),      #  to dim_regions
        df_invoices["invoice_date"],
        df_invoices["invoice_month"],
        df_invoices["invoice_status"],
        df_lines["currency"],
        df_lines["quantity"],
        df_lines["unit_price"],
        df_lines["discount"],
        df_lines["line_total"],
        df_lines["line_total_usd"]                       # Pre-converted to USD
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("sales_id")

df_fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.fact_sales")

print(f" {df_fact_sales.count()} rows")

In [0]:

print("Creating Fact Payments Table\n")

df_payments = spark.table("globaldistributor.02_silver.payments")
df_invoices = spark.table("globaldistributor.02_silver.invoices")

print("fact_payments")
# Join payments with invoices to get customer and region information
df_fact_payments = df_payments \
    .join(df_invoices, "invoice_id", "inner") \
    .select(
        df_payments["payment_id"],                    # Primary key
        df_payments["invoice_id"],                     # FK to dim_invoices
        df_invoices["customer_id"],                    # FK to dim_customers
        df_invoices["region"].alias("region_code"),    # FK to dim_regions
        df_payments["payment_date"],
        df_payments["payment_amount"],
        df_payments["payment_method"],
        df_invoices["currency"],
        df_invoices["invoice_status"]
    ) \
    .withColumn("gold_updated_at", current_timestamp()) \
    .orderBy("payment_id")

df_fact_payments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.fact_payments")

print(f"{df_fact_payments.count()} rows")


In [0]:
print("Creating Data Cube\n")

# Load fact and dimension tables
df_fact = spark.table("globaldistributor.03_gold.fact_sales")
df_customers = spark.table("globaldistributor.03_gold.dim_customers")
df_products = spark.table("globaldistributor.03_gold.dim_products")
df_regions = spark.table("globaldistributor.03_gold.dim_regions")

print("Joining fact with dimensions")

# Create denormalized data cube with all dimensional attributes
df_data_cube = df_fact \
    .join(df_customers, "customer_id", "left") \
    .join(df_products, "product_id", "left") \
    .join(df_regions, df_fact["region_code"] == df_regions["region_code"], "left") \
    .select(
        # Fact measures
        df_fact["sales_id"],
        df_fact["invoice_id"],
        df_fact["invoice_date"],
        df_fact["invoice_month"],
        df_fact["invoice_status"],
        df_fact["currency"],
        df_fact["quantity"],
        df_fact["unit_price"],
        df_fact["discount"],
        df_fact["line_total"],
        df_fact["line_total_usd"],
        # Customer dimensions
        df_fact["customer_id"],
        df_customers["customer_name"],
        df_customers["customer_type"],
        df_customers["country"].alias("customer_country"),
        df_customers["signup_date"],
        # Product dimensions
        df_fact["product_id"],
        df_products["product_name"],
        df_products["category"],
        df_products["cost_price"],
        # Region dimensions
        df_fact["region_code"],
        df_regions["region_name"],
        df_regions["country"].alias("region_country")
    ) \
    .orderBy("sales_id")

print(f"Data cube created: {df_data_cube.count()} rows")

# Persist the data cube as a table for direct querying
print("Saving data cube as table")
df_data_cube.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.data_cube")

print("Data cube saved as: globaldistributor.03_gold.data_cube")


In [0]:
display(spark.table("globaldistributor.03_gold.data_cube"))

In [0]:
# KPI 1: Total Revenue (USD)

print("KPI 1: Total Revenue (USD)")

df_kpi_total_revenue = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd"))

df_kpi_total_revenue.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_total_revenue")
display(df_kpi_total_revenue)

In [0]:

print("KPI 2: Total Quantity Sold")

df_kpi_qty = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .agg(spark_sum("quantity").alias("total_quantity_sold"))

df_kpi_qty.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_total_quantity")
display(df_kpi_qty)

In [0]:

print("KPI 3: Average Invoice Value (USD)")

df_invoice_totals = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("invoice_id") \
    .agg(spark_sum("line_total_usd").alias("invoice_total_usd"))

df_kpi_avg_invoice = df_invoice_totals \
    .agg(spark_round(avg("invoice_total_usd"), 2).alias("avg_invoice_value_usd"))

df_kpi_avg_invoice.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_avg_invoice_value")
display(df_kpi_avg_invoice)

In [0]:

# KPI 4: Top 20 Customers by Revenue

print("KPI 4: Top 20 Customers by Revenue")
df_kpi_top_customers = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("customer_id", "customer_name", "customer_type", "customer_country") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd")) \
    .orderBy(col("total_revenue_usd").desc()) \
    .limit(20)

df_kpi_top_customers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_top_customers")
display(df_kpi_top_customers)

In [0]:

# KPI 5: Top 20 Products by Revenue

print("KPI 5: Top 20 Products by Revenue...")

# No need to join with dim_products - already in data cube
df_kpi_top_products = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("product_id", "product_name", "category") \
    .agg(
        spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd"),
        spark_sum("quantity").alias("total_qty_sold")
    ) \
    .orderBy(col("total_revenue_usd").desc()) \
    .limit(20)

df_kpi_top_products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_top_products")
display(df_kpi_top_products)

In [0]:

# KPI 6: Invoice Cancellation Rate
print("KPI 6: Invoice Cancellation Rate")

# Calculate from invoices in the data cube
df_kpi_cancellation = df_data_cube \
    .select("invoice_id", "invoice_status") \
    .dropDuplicates(["invoice_id"]) \
    .agg(
        count("invoice_id").alias("total_invoices"),
        spark_sum(when(col("invoice_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_invoices")
    ) \
    .withColumn(
        "cancellation_rate_pct",
        spark_round((col("cancelled_invoices") / col("total_invoices")) * 100, 2)
    )

df_kpi_cancellation.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_cancellation_rate")
display(df_kpi_cancellation)

In [0]:

# KPI 7: Discount Impact per Customer

print("KPI 7: Discount Impact per Customer")

# No need to join - customer info already in data cube
df_kpi_discount = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("customer_id", "customer_name") \
    .agg(
        spark_round(spark_sum("discount"), 2).alias("total_discount_given"),
        spark_round(avg("discount"), 2).alias("avg_discount_per_line_item")
    ) \
    .orderBy(col("total_discount_given").desc()) \
    .limit(100)

df_kpi_discount.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_discount_impact")
display(df_kpi_discount.limit(20))

In [0]:

# KPI 8: Revenue by Region

print("KPI 8: Revenue by Region")

# Region info already in data cube
df_kpi_region = df_data_cube \
    .filter(col("invoice_status") != "CANCELLED") \
    .groupBy("region_code") \
    .agg(spark_round(spark_sum("line_total_usd"), 2).alias("total_revenue_usd")) \
    .orderBy(col("total_revenue_usd").desc())

df_kpi_region.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_revenue_by_region")
display(df_kpi_region)

In [0]:

# KPI 9: Number of Invoices per Customer

print("KPI 9: Invoices per Customer (Purchase Frequency)")

# Calculate from data cube
df_kpi_invoice_freq = df_data_cube \
    .groupBy("customer_id", "customer_name") \
    .agg(countDistinct("invoice_id").alias("invoice_count")) \
    .orderBy(col("invoice_count").desc()) \
    .limit(100)

df_kpi_invoice_freq.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globaldistributor.03_gold.kpi_invoices_per_customer")

display(df_kpi_invoice_freq.limit(20))



In [0]:




print("KPI Tables:\n")

# KPI 1
print("1. TOTAL REVENUE (USD)")
kpi1 = spark.table("globaldistributor.03_gold.kpi_total_revenue").collect()[0][0]
print(f"   ${kpi1:,.2f}\n")

# KPI 2
print("2. TOTAL QUANTITY SOLD")
kpi2 = spark.table("globaldistributor.03_gold.kpi_total_quantity").collect()[0][0]
print(f"   {kpi2:,} units\n")

# KPI 3
print("3. AVERAGE INVOICE VALUE (USD)")
kpi3 = spark.table("globaldistributor.03_gold.kpi_avg_invoice_value").collect()[0][0]
print(f"   ${kpi3:,.2f}\n")

# KPI 4
print("4. TOP 5 CUSTOMERS BY REVENUE")
spark.table("globaldistributor.03_gold.kpi_top_customers").limit(5).display()

# KPI 5
print("\n5. TOP 5 PRODUCTS BY REVENUE")
spark.table("globaldistributor.03_gold.kpi_top_products").limit(5).display()

# KPI 6
print("\n6. INVOICE CANCELLATION RATE")
kpi6 = spark.table("globaldistributor.03_gold.kpi_cancellation_rate")
kpi6.display()

# KPI 7
print("\n7. TOP 5 CUSTOMERS BY DISCOUNT RECEIVED")
spark.table("globaldistributor.03_gold.kpi_discount_impact").limit(5).display()

# KPI 8
print("\n8. REVENUE BY REGION")
spark.table("globaldistributor.03_gold.kpi_revenue_by_region").display()

# KPI 9
print("\n9. TOP 5 CUSTOMERS BY PURCHASE FREQUENCY")
spark.table("globaldistributor.03_gold.kpi_invoices_per_customer").limit(5).display()

